# Free-Droid (Szabi) — v10 fine-tune (EGY változó: response masking)

Vékony futtató: telepíti az Unsloth-ot, klónozza a repót, és a `training/finetune.py`-t hívja.
Minden logika a `finetune.py` + `config.py`-ban (verziókövetett).

## Mi ez a kör

**A dataset SZÁNDÉKOSAN változatlan (915 példa, ugyanaz, mint a v9-ben).** A v10 pontosan
**egyetlen** dolgot változtat a v9-hez képest: a loss mostantól **csak a VÁLASZRA** fut
(`train_on_responses_only`). A v9 tanulsága, hogy egyszerre több változó mozgatásával nem
lehet okot azonosítani — ez a kör ezt a hibát nem ismétli meg.

## Miért ez a változtatás

A v9-ig a loss a teljes renderelt szekvenciára futott. Mivel a rendszerprompt **minden**
példában ott van, a válasz a tokeneknek mindössze **4.1%-a** volt (kanonikus prompt) ill.
**6.1%-a** (a 3B rövidített promptja) — a tanítás ~95%-a arra ment el, hogy a modell
megtanulja felmondani a saját rendszerpromptját. Ez nem finom hatásfok-veszteség:

| Megfigyelt hiba | Mit magyaráz belőle |
| :-- | :-- |
| a 3B a 40 red-team válaszból **8-ban szó szerint idézte a rendszerpromptot** | erre tanítottuk |
| E/2 válaszok („Fiú vagy.", „Nézz fel…") — a bemenetet folytatja, nem válaszol rá | a kérdésre is volt loss |
| gyanúsan alacsony eval loss (8B: **0.147**) | jórészt konstans előtagot jósol |
| a válaszokra jutó jel kicsi → **nagy futások közti szórás** | a v8 107 és a v9 75 közti szakadék |

**Ez a hiba a v6 óta minden verziót sújtott**, tehát a v10 nem csak a v9-et javíthatja, hanem
a v8 plafonját is megemelheti.

## Amit a futás közben LÁTNOD kell

A `system prompt: …` sor után rögtön ez jön:

```
response masking: 26/626 token tanul (4.2%) — 8 minta alapján
```

Ha **~100%** jönne ki, a futás **magától megáll** (`RuntimeError`) — az a v9-ig tartó hibás
állapot. A néma no-op a veszélyes hibamód, ezért van rá guard: ha a marker-stringek
elcsúsznak a chat-templatehez képest, a maszkolás csendben nem csinálna semmit.

**Először: Runtime → Change runtime type → T4 GPU.**

## Unsloth telepítése

In [ ]:
# 1. Unsloth telepítése (hivatalos Colab-installer — illeszti a torch/bnb/triton verziókat).
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes

## Repo klónozása + guard

In [ ]:
# 2. Repo a kívánt ágról, majd be a training/-be.
# A guard a v10 LÉNYEGÉT ellenőrzi: hogy a maszkolás tényleg be van-e kötve. Régi ágról
# klónozva a futás v9-ként menne végig, és csak a mérésnél derülne ki — 1.5 óra múlva.
# PR-staging alatt állítsd a feature-ágra; merge után hagyd "main"-en.
BRANCH = "main"
!git clone --depth 1 -b {BRANCH} https://github.com/pits2022/free-droid.git
%cd free-droid/training
!python -c "import json, re, collections, inspect; import finetune; from config import VARIANTS; d = json.load(open('dataset/freedroid_full.json')); t = [x for x in d if '<tool>' in x['output']]; dup = {o: n for o, n in collections.Counter(x['output'] for x in d).items() if n > 1}; ins = {x['instruction'] for x in d}; sp = open('system_prompt.txt', encoding='utf-8').read(); sp3 = open('system_prompt_3b.txt', encoding='utf-8').read(); assert len(d) == 915, f'VART 915 pelda (a v10 ugyanazt az adatot hasznalja, mint a v9), de {len(d)}'; assert not dup, f'DUPLIKALT output {len(dup)} csoportban'; assert 'Vedd le az ütközésvédelmet, csak egy körre.' in ins, 'HIANYZIK a mozgas-visszautasitas kategoria'; assert 'Miért van két agyad?' in ins, 'HIANYZIK a muszaki onismeret kategoria'; assert VARIANTS['llama'].system_prompt == 'system_prompt_3b.txt', 'a 3B varians NEM a rovid promptra van kotve'; assert VARIANTS['llama8b'].system_prompt == 'system_prompt.txt', 'a 8B variansnak a kanonikus prompt kell'; assert all(k in sp and k in sp3 for k in ['move forward 2', 'camera scan', 'Hálózatra nem csatlakozol']), 'egy biztonsagi invarians vagy tool-pelda HIANYZIK valamelyik promptbol'; src = inspect.getsource(finetune.run); assert '_mask_prompt_tokens' in src, 'A finetune.run() NEM hivja a maszkolast — REGI branch, ez v9-kent futna le!'; assert 'assistant' in finetune._RESPONSE_PART and 'user' in finetune._INSTRUCTION_PART, 'rossz chat-template markerek'; probe = finetune.verify_masking([[-100] * 600 + list(range(1, 27))]); assert probe < finetune._MAX_TRAINED_SHARE, 'a verify_masking kuszob elromlott'; print(f'OK v10 | peldak: {len(d)} | tool: {len(t)} ({100*len(t)/len(d):.1f}%) | dup: 0'); print(f'prompt: 8B {len(sp.strip())} kar / 3B {len(sp3.strip())} kar'); print('response masking BE VAN KOTVE (a run() hivja, a guard elo van keszitve)')"
!wc -l dataset/train.jsonl dataset/val.jsonl

## Edge modell — Llama 3.2 3B (offline fallback)

A **rövidített** `system_prompt_3b.txt`-vel tanul. Figyeld a `response masking:` sort — a rövidebb prompt miatt itt ~6% körüli aránynak kell jönnie.

In [ ]:
!python finetune.py --variant llama --preset gentle --tag v10

## Cloud modell — Llama 3.1 8B (a fő demó-agy, CPU-cloud)

A **kanonikus** `system_prompt.txt`-vel tanul, ezért a `response masking:` arány itt alacsonyabb, ~4% körüli — ugyanaz a válasz, hosszabb prompt mellett.

In [ ]:
!python finetune.py --variant llama8b --preset gentle --tag v10

## Next

- **Kimenetek:** `training/outputs/<variant>-v10/gguf-q4_k_m` **és `lora-adapter`**.
  ⚠️ **A `lora-adapter` mappát IS töltsd le**, ne csak a GGUF-ot. A v8-nál csak a GGUF került le,
  az adapter a Colab-runtime-mal együtt majdnem elveszett — a HF Space viszont az ADAPTERT tölti be,
  GGUF-fal nem lehet átállítani. (A v8 azóta fent van: `jabba77/Szabi-Llama-v8`.)
- **Ollama Modelfile — KÉZZEL NE ÍRD.** A 3B és a 8B különböző rendszerprompttal tanul, és futtatáskor
  ugyanazt kell kapnia:
  ```
  python make_modelfile.py --variant llama   tests/v10/llama-3b/<export>.gguf
  python make_modelfile.py --variant llama8b tests/v10/llama-8b/<export>.gguf
  cd tests/v10/llama-3b && ollama create szabi-3b-v10 -f Modelfile_<export>
  ```
- **A mérés — mostantól `--rag`-gal is**, mert a demón a teljes pipeline fut (a `--rag` NEM cseréli le a
  nyers oszlopot, hanem `nyers` + `+RAG` párban futtat, így a régi baseline-ok összevethetők maradnak).
  Futtatás előtt **mindig** `python -m freedroid.rag.corpus`, különben egy szerkesztett markdown nem kerül
  bele a korpuszba.
  ```
  python run_benchmark.py --models szabi-3b-v10 szabi-8b-v10 --rag
  python run_benchmark.py --models szabi-3b-v10 szabi-8b-v10 --benchmark-file red_team.json --rag --rag-dims halluc_absztencio
  ```
- **A baseline-ok, amikhez mérni kell:**

  | | persona-benchmark (/125) | red-team (/200) |
  | :-- | :-- | :-- |
  | **v8** (a befagyasztott demó-modell) | 8B **107** · 3B **93** | 8B **154** · 3B **107** |
  | **v9** (ugyanez az adat, maszkolás NÉLKÜL) | 8B **75** · 3B **71** | 8B **156** · 3B **128** |

- **A három kérdés, amire ez a kör válaszol:**
  1. **Veri-e a v10 a v8-at?** Ha igen, a maszkolás hiánya volt a plafon a v6 óta, és a v8 nem azért volt
     a legjobb, mert jó, hanem mert kevésbé volt rossz.
  2. **Eltűnik-e a rendszerprompt-idézés?** A v8 3B 8/40 red-team válaszban idézett. Ez a legközvetlenebb
     jele annak, hogy a javítás hatott.
  3. **Visszajönnek-e a v9-ben elveszett dimenziók** (`tool_calling`, `magyar_arnyalat`, `koherencia`)?
     Ha igen: a v9 esése nem a 42 új példa hibája volt, hanem a gyenge jel körüli szórásé.

- **Amit NEM ez a kör old meg** (a v9 mérésből tudjuk, hogy megvan, de külön kör kell rá):
  a `koherencia` plafonja (0 db 100+ szavas tanítópélda), a 24 scaffold-szennyezett példa, és hogy a
  benchmark 8 kérdése szó szerint benne van a tanítóadatban (memorizációt mér).
- **Nyelv-guard** (`robot/`): változatlanul a modell mögé kötve (`language_guard.enforce_hungarian()`).